# Analysis

Analysis-only notebook, separated from `prompt_optimisation.ipynb` so you can explore `results_log.jsonl` while a sweep is still running there. This notebook never calls the model — it only reads `notebook_results/results_log.jsonl` via `result_logger.load_results()`.

In [1]:
import pandas as pd

from analysis_util import compute_binary_metrics
from result_logger import load_results

## Review logged results

Load the full log (all batches) or just one `batch_id` to compare prompts/models without re-running anything.

In [2]:
df_all = load_results()  # or load_results(batch_id="batch1")
df_all['timestamp'] = pd.to_datetime(df_all['timestamp'])

In [3]:
df_all['model'].value_counts()

model
gemini-2.5-pro            477
gemini-3.5-flash          398
gemini-3.1-pro-preview    322
gemini-2.5-flash          108
Name: count, dtype: int64

In [4]:
df_all.columns

Index(['timestamp', 'batch_id', 'image_path', 'prompt_id', 'model',
       'temperature', 'latency_s', 'raw_response', 'parsed_response', 'notes',
       'label', 'label_signals', 'verdict', 'confidence', 'signal_types',
       'format', 'augmentation'],
      dtype='object')

In [5]:
df_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1305 entries, 0 to 1304
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   timestamp        1305 non-null   datetime64[ns, UTC]
 1   batch_id         1305 non-null   object             
 2   image_path       1305 non-null   object             
 3   prompt_id        1305 non-null   object             
 4   model            1305 non-null   object             
 5   temperature      1305 non-null   float64            
 6   latency_s        1305 non-null   float64            
 7   raw_response     1305 non-null   object             
 8   parsed_response  1294 non-null   object             
 9   notes            1305 non-null   object             
 10  label            1305 non-null   object             
 11  label_signals    1305 non-null   object             
 12  verdict          1294 non-null   object             
 13  confidence       1

In [6]:
# Calculate some accuracy, recall, F1 metrics
total_tests = len(df_all)
total_wrong_verdict = len(df_all[df_all['label'] != df_all['verdict']])

print("Wrong_verdict_percent: ", total_wrong_verdict/total_tests*100)

Wrong_verdict_percent:  28.50574712643678


### Different batches will have different performances

In [7]:
df_all['verdict'].value_counts()

verdict
tampered        724
authentic       569
inconclusive      1
Name: count, dtype: int64

In [8]:
total_matrics = compute_binary_metrics(df_all)
print(total_matrics)

precision           0.748619
recall              0.752778
specificity         0.682373
f1                  0.750693
accuracy            0.721578
true_positive     542.000000
false_positive    182.000000
false_negative    178.000000
true_negative     391.000000
excluded_count     12.000000
dtype: float64


#### Batch 1

In [9]:
batch1 = df_all[df_all['batch_id'] == 'batch1']
print(compute_binary_metrics(batch1))

precision           0.964539
recall              0.814371
specificity         0.861111
f1                  0.883117
accuracy            0.822660
true_positive     136.000000
false_positive      5.000000
false_negative     31.000000
true_negative      31.000000
excluded_count      0.000000
dtype: float64


#### Batch 2

In [10]:
batch2 = df_all[df_all['batch_id'] == 'batch2']
print(compute_binary_metrics(batch2))

precision           0.668605
recall              0.583756
specificity         0.695187
f1                  0.623306
accuracy            0.638021
true_positive     115.000000
false_positive     57.000000
false_negative     82.000000
true_negative     130.000000
excluded_count      2.000000
dtype: float64


#### Compare different prompt versions

In [11]:
df_all['prompt_id'].value_counts()

prompt_id
V2          584
V2_split    377
V1          178
V1_split    166
Name: count, dtype: int64

In [12]:
df_all.groupby('prompt_id').apply(compute_binary_metrics)

/var/folders/1k/s8c8mkvd19307rbdq3mhs23h0000gp/T/ipykernel_18972/994550254.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_all.groupby('prompt_id').apply(compute_binary_metrics)


,precision,recall,specificity,f1,accuracy,true_positive,false_positive,false_negative,true_negative,excluded_count
prompt_id,,,,,,,,,,
V1,0.820225,0.657658,0.753846,0.730000,0.693182,73.0,16.0,38.0,49.0,2.0
V1_split,0.802632,0.575472,0.750000,0.670330,0.638554,61.0,15.0,45.0,45.0,0.0
V2,0.749263,0.832787,0.685185,0.788820,0.763478,254.0,85.0,51.0,185.0,9.0
V2_split,0.700000,0.777778,0.629213,0.736842,0.707447,154.0,66.0,44.0,112.0,1.0


In [13]:
# Latency check: Latency is better?, To check again when have more results
print(df_all.groupby(['prompt_id'])['latency_s'].mean())
print(df_all.groupby(['prompt_id'])['latency_s'].std())

prompt_id
V1          16.808764
V1_split    15.575602
V2          23.634966
V2_split    22.217825
Name: latency_s, dtype: float64
prompt_id
V1           9.504234
V1_split     7.307306
V2          12.375832
V2_split    21.104867
Name: latency_s, dtype: float64


#### Model performances

In [14]:
df_all.groupby('model').apply(compute_binary_metrics)

/var/folders/1k/s8c8mkvd19307rbdq3mhs23h0000gp/T/ipykernel_18972/3643566522.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_all.groupby('model').apply(compute_binary_metrics)


,precision,recall,specificity,f1,accuracy,true_positive,false_positive,false_negative,true_negative,excluded_count
model,,,,,,,,,,
gemini-2.5-flash,0.638889,0.370968,0.711111,0.469388,0.514019,23.0,13.0,39.0,32.0,1.0
gemini-2.5-pro,0.670270,0.885714,0.361257,0.763077,0.673036,248.0,122.0,32.0,69.0,6.0
gemini-3.1-pro-preview,0.792899,0.797619,0.769737,0.795252,0.784375,134.0,35.0,34.0,117.0,2.0
gemini-3.5-flash,0.919463,0.652381,0.935135,0.763231,0.784810,137.0,12.0,73.0,173.0,3.0


In [15]:
# Latency check: Latency is better?, To check again when have more results
print("Latency mean: ", df_all.groupby(['model'])['latency_s'].mean())
print("Latency std: ", df_all.groupby(['model'])['latency_s'].std())

Latency mean:  model
gemini-2.5-flash          15.915278
gemini-2.5-pro            24.661279
gemini-3.1-pro-preview    27.582019
gemini-3.5-flash          13.549648
Name: latency_s, dtype: float64
Latency std:  model
gemini-2.5-flash           8.560064
gemini-2.5-pro             6.710681
gemini-3.1-pro-preview    25.481962
gemini-3.5-flash           5.324114
Name: latency_s, dtype: float64


In [16]:
batch1.groupby('model').apply(compute_binary_metrics)

/var/folders/1k/s8c8mkvd19307rbdq3mhs23h0000gp/T/ipykernel_18972/3724333011.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  batch1.groupby('model').apply(compute_binary_metrics)


,precision,recall,specificity,f1,accuracy,true_positive,false_positive,false_negative,true_negative,excluded_count
model,,,,,,,,,,
gemini-2.5-flash,1.000000,0.458333,1.000000,0.628571,0.566667,11.0,0.0,13.0,6.0,0.0
gemini-2.5-pro,0.948454,0.968421,0.772727,0.958333,0.931624,92.0,5.0,3.0,17.0,0.0
gemini-3.1-pro-preview,1.000000,0.708333,1.000000,0.829268,0.750000,17.0,0.0,7.0,4.0,0.0
gemini-3.5-flash,1.000000,0.666667,1.000000,0.800000,0.714286,16.0,0.0,8.0,4.0,0.0


In [17]:
batch2.groupby('model').apply(compute_binary_metrics)

/var/folders/1k/s8c8mkvd19307rbdq3mhs23h0000gp/T/ipykernel_18972/2769963313.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  batch2.groupby('model').apply(compute_binary_metrics)


,precision,recall,specificity,f1,accuracy,true_positive,false_positive,false_negative,true_negative,excluded_count
model,,,,,,,,,,
gemini-2.5-flash,0.588235,0.588235,0.611111,0.588235,0.600000,10.0,7.0,7.0,11.0,1.0
gemini-2.5-pro,0.571429,0.733333,0.326531,0.642336,0.550459,44.0,33.0,16.0,16.0,1.0
gemini-3.1-pro-preview,0.736842,0.700000,0.750000,0.717949,0.725000,42.0,15.0,18.0,45.0,0.0
gemini-3.5-flash,0.904762,0.316667,0.966667,0.469136,0.641667,19.0,2.0,41.0,58.0,0.0


### Check Temperature parameter performances



In [18]:
df_all.groupby('temperature').apply(compute_binary_metrics)

/var/folders/1k/s8c8mkvd19307rbdq3mhs23h0000gp/T/ipykernel_18972/547241242.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_all.groupby('temperature').apply(compute_binary_metrics)


,precision,recall,specificity,f1,accuracy,true_positive,false_positive,false_negative,true_negative,excluded_count
temperature,,,,,,,,,,
0.00,0.756098,0.607843,0.750000,0.673913,0.670330,31.0,10.0,20.0,30.0,0.0
0.05,0.750000,0.545455,0.714286,0.631579,0.611111,12.0,4.0,10.0,10.0,1.0
0.10,0.746967,0.789377,0.667426,0.767587,0.735025,431.0,146.0,115.0,293.0,11.0
0.20,0.760870,0.686275,0.725000,0.721649,0.703297,35.0,11.0,16.0,29.0,0.0
0.80,0.750000,0.660000,0.725000,0.702128,0.688889,33.0,11.0,17.0,29.0,0.0


In [19]:
batch1.groupby('temperature').apply(compute_binary_metrics)

/var/folders/1k/s8c8mkvd19307rbdq3mhs23h0000gp/T/ipykernel_18972/1052639328.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  batch1.groupby('temperature').apply(compute_binary_metrics)


,precision,recall,specificity,f1,accuracy,true_positive,false_positive,false_negative,true_negative,excluded_count
temperature,,,,,,,,,,
0.00,1.000000,0.933333,1.000000,0.965517,0.947368,14.0,0.0,1.0,4.0,0.0
0.05,1.000000,0.454545,1.000000,0.625000,0.538462,5.0,0.0,6.0,2.0,0.0
0.10,0.956989,0.794643,0.818182,0.868293,0.798507,89.0,4.0,23.0,18.0,0.0
0.20,1.000000,1.000000,1.000000,1.000000,1.000000,15.0,0.0,0.0,4.0,0.0
0.80,0.928571,0.928571,0.750000,0.928571,0.888889,13.0,1.0,1.0,3.0,0.0


In [20]:
batch2.groupby('temperature').apply(compute_binary_metrics)

/var/folders/1k/s8c8mkvd19307rbdq3mhs23h0000gp/T/ipykernel_18972/547609140.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  batch2.groupby('temperature').apply(compute_binary_metrics)


,precision,recall,specificity,f1,accuracy,true_positive,false_positive,false_negative,true_negative,excluded_count
temperature,,,,,,,,,,
0.00,0.629630,0.472222,0.722222,0.539683,0.597222,17.0,10.0,19.0,26.0,0.0
0.05,0.636364,0.636364,0.666667,0.636364,0.652174,7.0,4.0,4.0,8.0,1.0
0.10,0.698630,0.653846,0.671642,0.675497,0.662069,51.0,22.0,27.0,45.0,1.0
0.20,0.645161,0.555556,0.694444,0.597015,0.625000,20.0,11.0,16.0,25.0,0.0
0.80,0.666667,0.555556,0.722222,0.606061,0.638889,20.0,10.0,16.0,26.0,0.0


In [23]:
augmented = load_results(batch_id = 'augmented_slips')
augmented.groupby('temperature').apply(compute_binary_metrics)

/var/folders/1k/s8c8mkvd19307rbdq3mhs23h0000gp/T/ipykernel_18972/2768621478.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  augmented.groupby('temperature').apply(compute_binary_metrics)


,precision,recall,specificity,f1,accuracy,true_positive,false_positive,false_negative,true_negative,excluded_count
temperature,,,,,,,,,,
0.1,0.680628,0.77381,0.651429,0.724234,0.71137,130.0,61.0,38.0,114.0,1.0


#### Stability analysis
- Let's see how often do each model/ per batch/ per temperature change their predictions

In [15]:
df_all['image_path'].value_counts()

image_path
test_images/payslip2.png                                               62
test_images/payslip1.jpeg                                              62
test_images/payslip30.png                                              62
test_images/payslip3.png                                               62
test_images/payslip20.png                                              62
test_images/payslip10.jpeg                                             62
test_images/first-tamper.png                                           38
test_images/First_Tamper.png                                           38
test_images/combined.png                                               36
test_images/pink.png                                                   34
test_images/neat.png                                                   31
test_images/cropped-obvious.png                                        20
test_images/neatest.png                                                20
test_images_augmented/by_te

In [31]:
verdict_proportions = df_all.groupby('image_path')['verdict'].value_counts(normalize=True)
verdict_proportions

image_path                                                           verdict     
test_images/First_Tamper.png                                         authentic       0.868421
                                                                     tampered        0.131579
test_images/combined.png                                             tampered        1.000000
test_images/cropped-obvious.png                                      tampered        0.900000
                                                                     authentic       0.100000
test_images/first-tamper.png                                         tampered        0.894737
                                                                     authentic       0.105263
test_images/neat.png                                                 tampered        0.580645
                                                                     authentic       0.419355
test_images/neatest.png                                              aut

In [17]:
batch2.groupby('image_path').apply(compute_binary_metrics)

/var/folders/1k/s8c8mkvd19307rbdq3mhs23h0000gp/T/ipykernel_17219/2644542247.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  batch2.groupby('image_path').apply(compute_binary_metrics)


,precision,recall,specificity,f1,accuracy,true_positive,false_positive,false_negative,true_negative,excluded_count
image_path,,,,,,,,,,
test_images/First_Tamper.png,NaN,NaN,1.000000,NaN,1.000000,0.0,0.0,0.0,2.0,0.0
test_images/combined.png,1.0,1.000000,NaN,1.000000,1.000000,2.0,0.0,0.0,0.0,0.0
test_images/cropped-obvious.png,1.0,1.000000,NaN,1.000000,1.000000,2.0,0.0,0.0,0.0,0.0
test_images/first-tamper.png,1.0,1.000000,NaN,1.000000,1.000000,2.0,0.0,0.0,0.0,0.0
test_images/neat.png,1.0,1.000000,NaN,1.000000,1.000000,2.0,0.0,0.0,0.0,0.0
test_images/neatest.png,1.0,1.000000,NaN,1.000000,1.000000,2.0,0.0,0.0,0.0,0.0
test_images/payslip1.jpeg,1.0,0.344262,NaN,0.512195,0.344262,21.0,0.0,40.0,0.0,1.0
test_images/payslip10.jpeg,0.0,NaN,0.770492,NaN,0.770492,0.0,14.0,0.0,47.0,1.0
test_images/payslip2.png,1.0,0.774194,NaN,0.872727,0.774194,48.0,0.0,14.0,0.0,0.0


### Check the augmented results
- Choose the slip images with strongest performance
- Augmented the slip images into the test_images_augmented/by_technique folder
- To augmented = load_results(batch = augmented_payslips) 
- Analyse to come up with V3 prompt/ find any weaknesses

In [22]:
augmented = load_results(batch_id = 'augmented_slips')

In [44]:
augmented.columns

Index(['timestamp', 'batch_id', 'image_path', 'prompt_id', 'model',
       'temperature', 'latency_s', 'raw_response', 'parsed_response', 'notes',
       'label', 'label_signals', 'verdict', 'confidence', 'signal_types',
       'format', 'augmentation'],
      dtype='object')

In [46]:
augmented['image_path'].value_counts()

image_path
test_images_augmented/by_technique/payslip2_rotation.png                10
test_images_augmented/by_technique/payslip30_perspective.png            10
test_images_augmented/by_technique/payslip30_shadow_vignette.png        10
test_images_augmented/by_technique/payslip30_gaussian_blur.png          10
test_images_augmented/by_technique/payslip30_jpeg_compression.png       10
test_images_augmented/by_technique/payslip2_perspective.png             10
test_images_augmented/by_technique/payslip30_elastic_distortion.png     10
test_images_augmented/by_technique/payslip30_crop_and_pad.png           10
test_images_augmented/by_technique/payslip30_rotation.png               10
test_images_augmented/by_technique/payslip2_shadow_vignette.png         10
test_images_augmented/by_technique/payslip2_gaussian_blur.png           10
test_images_augmented/by_technique/payslip2_jpeg_compression.png        10
test_images_augmented/by_technique/payslip2_crop_and_pad.png            10
test_images_au

In [38]:
augmented.groupby('image_path').apply(compute_binary_metrics)

/var/folders/1k/s8c8mkvd19307rbdq3mhs23h0000gp/T/ipykernel_1524/3519723449.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  augmented.groupby('image_path').apply(compute_binary_metrics)


,precision,recall,specificity,f1,accuracy,true_positive,false_positive,false_negative,true_negative,excluded_count
image_path,,,,,,,,,,
test_images_augmented/by_technique/payslip2_crop_and_pad.png,1.0,0.625,NaN,0.769231,0.625,5.0,0.0,3.0,0.0,0.0
test_images_augmented/by_technique/payslip2_elastic_distortion.png,1.0,0.750,NaN,0.857143,0.750,6.0,0.0,2.0,0.0,0.0
test_images_augmented/by_technique/payslip2_gaussian_blur.png,1.0,0.750,NaN,0.857143,0.750,6.0,0.0,2.0,0.0,0.0
test_images_augmented/by_technique/payslip2_jpeg_compression.png,1.0,0.500,NaN,0.666667,0.500,4.0,0.0,4.0,0.0,0.0
test_images_augmented/by_technique/payslip2_perspective.png,1.0,0.750,NaN,0.857143,0.750,6.0,0.0,2.0,0.0,0.0
test_images_augmented/by_technique/payslip2_rotation.png,1.0,0.750,NaN,0.857143,0.750,6.0,0.0,2.0,0.0,0.0
test_images_augmented/by_technique/payslip2_shadow_vignette.png,1.0,0.750,NaN,0.857143,0.750,6.0,0.0,2.0,0.0,0.0
test_images_augmented/by_technique/payslip30_crop_and_pad.png,0.0,NaN,0.500,NaN,0.500,0.0,4.0,0.0,4.0,0.0
test_images_augmented/by_technique/payslip30_elastic_distortion.png,0.0,NaN,0.625,NaN,0.625,0.0,3.0,0.0,5.0,0.0


In [39]:
augmented.groupby('model').apply(compute_binary_metrics)

/var/folders/1k/s8c8mkvd19307rbdq3mhs23h0000gp/T/ipykernel_1524/3223642045.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  augmented.groupby('model').apply(compute_binary_metrics)


,precision,recall,specificity,f1,accuracy,true_positive,false_positive,false_negative,true_negative,excluded_count
model,,,,,,,,,,
gemini-2.5-flash,0.333333,0.071429,0.857143,0.117647,0.464286,1.0,2.0,13.0,12.0,0.0
gemini-2.5-pro,0.416667,0.714286,0.000000,0.526316,0.357143,10.0,14.0,4.0,0.0,0.0
gemini-3.1-pro-preview,0.933333,1.000000,0.928571,0.965517,0.964286,14.0,1.0,0.0,13.0,0.0
gemini-3.5-flash,0.823529,1.000000,0.785714,0.903226,0.892857,14.0,3.0,0.0,11.0,0.0


In [ ]:
augmented.groupby(['model', 'prompt-library'])

### Overview report

In [32]:
BATCH_ID = 'slip2'
slip2 = load_results(batch_id= BATCH_ID)
slips_result = slip2.groupby(['model', 'prompt_id','temperature', 'image_path']).apply(compute_binary_metrics)
latent = slip2.groupby(['model', 'prompt_id','temperature']).agg(
    latency_mean=('latency_s', 'mean'),latency_std=('latency_s', 'std'), count=('latency_s', 'count'))
latent

/var/folders/1k/s8c8mkvd19307rbdq3mhs23h0000gp/T/ipykernel_18972/3423702653.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  slips_result = slip2.groupby(['model', 'prompt_id','temperature', 'image_path']).apply(compute_binary_metrics)


,,,latency_mean,latency_std,count
model,prompt_id,temperature,,,
gemini-2.5-pro,V2,0.1,30.042917,7.062627,24
gemini-3.1-pro-preview,V2,0.1,36.807083,14.384726,24
gemini-3.5-flash,V2,0.1,15.827917,4.715835,24


In [25]:
slip2.columns

Index(['timestamp', 'batch_id', 'image_path', 'prompt_id', 'model',
       'temperature', 'latency_s', 'raw_response', 'parsed_response', 'notes',
       'label', 'label_signals', 'verdict', 'confidence', 'signal_types',
       'format', 'augmentation'],
      dtype='object')

In [34]:
test = slips_result[['accuracy']]
pivot_df = test['accuracy'].unstack(level='image_path')
# Save number of the image original + augmentation this way
ori_augment_number = len(pivot_df.columns)
print(ori_augment_number)
pivot_df[['latency_mean', 'latency_std']] = latent[['latency_mean', 'latency_std']]
pivot_df

4


,,image_path,test_images/payslip2.png,test_images_augmented/by_technique/payslip2_gaussian_blur.png,test_images_augmented/by_technique/payslip2_jpeg_compression.png,test_images_augmented/by_technique/payslip2_rotation.png,latency_mean,latency_std
model,prompt_id,temperature,,,,,,
gemini-2.5-pro,V2,0.1,0.666667,1.0,1.0,1.0,30.042917,7.062627
gemini-3.1-pro-preview,V2,0.1,0.833333,1.0,1.0,1.0,36.807083,14.384726
gemini-3.5-flash,V2,0.1,1.000000,1.0,1.0,1.0,15.827917,4.715835


In [ ]:
pivot_df.to_csv(BATCH_ID)

In [27]:
# list(slips_result.index)

In [35]:
for BATCH_ID in ["slip1", "slip2", "slip3", "slip10", "slip20", "slip30"]:
    df = load_results(batch_id=BATCH_ID)
    if df.empty:
        print(f"{BATCH_ID}: no data, skipping.")
        continue

    slips_result = df.groupby(['model', 'prompt_id', 'temperature', 'image_path']).apply(
        compute_binary_metrics, include_groups=False
    )
    latent = df.groupby(['model', 'prompt_id', 'temperature']).agg(
        latency_mean=('latency_s', 'mean'),
        latency_std=('latency_s', 'std'),
        count=('latency_s', 'count')
    )
    pivot_df = slips_result[['accuracy']].unstack(level='image_path')
    ori_augment_number = len(pivot_df.columns)
    pivot_df.columns = pivot_df.columns.droplevel(0)
    pivot_df[['latency_mean', 'latency_std', 'count']] = latent[['latency_mean', 'latency_std', 'count']]
    # Hardcode the fact that count of the group = count of the total sum/ # of augment
    pivot_df['count'] = pivot_df['count']/ori_augment_number

    pivot_df.to_csv(f"notebook_results/{BATCH_ID}_overview.csv")
    print(f"{BATCH_ID}: saved {pivot_df.shape[0]} rows → notebook_results/{BATCH_ID}_overview.csv")

slip1: saved 3 rows → notebook_results/slip1_overview.csv
slip2: saved 3 rows → notebook_results/slip2_overview.csv
slip3: saved 3 rows → notebook_results/slip3_overview.csv
slip10: saved 3 rows → notebook_results/slip10_overview.csv
slip20: saved 3 rows → notebook_results/slip20_overview.csv
slip30: saved 3 rows → notebook_results/slip30_overview.csv
